# DNA activity prediction model

The pipeline contains:
1. k-mer feature models: Logistic Regression, Ridge Regression, LightGBM classifier and LightGBM regressor,
2. a reverse-complement CNN trained on the raw DNA sequence,
3. a final weighted ensemble for `predicted_is_active`,
4. a LightGBM regressor for `predicted_rna_dna_ratio`,

In [ ]:
import os
import json
import copy
import random
import warnings

import numpy as np
import pandas as pd

import joblib
import lightgbm as lgb

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    mean_squared_error,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names"
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


In [ ]:
# Use GPU if available

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: Tesla T4


## 1. Load dataset and inspect basic properties

In [ ]:
# Inspect the raw dataset

df = pd.read_csv("dataset.tsv", sep="\t")
display(df.head())
print(df["sequence"].str.len().describe())
print(df["is_active"].value_counts(normalize=True))
print(df["rna_dna_ratio"].describe())

,seq_id,sequence,rna_dna_ratio,is_active
0,DNasePeakNoPromoter24818,TCGGTTCACGCAATGTTGCATCAGTTGGGACCGTTGTTACTGACCT...,0.490,1
1,DNasePeakNoPromoter39544,TCGGTTCACGCAATGCGGCCTCCCAAAGTTCTAGGATTACAGGCGT...,0.129,1
2,DNasePeakNoPromoter21529,TCGGTTCACGCAATGAACTCTCTACCATGAGTCAAATCCAGCCCAT...,-0.776,0
3,DNasePeakNoPromoter44110,TCGGTTCACGCAATGACATCGAATTTCATGGTTGGCAGAGGTAAAG...,-0.668,0
4,DNasePeakNoPromoter54524,TCGGTTCACGCAATGACACAAACTAAGGACTGACCCGTTGTGAGAT...,-0.638,0


count    55386.0
mean       230.0
std          0.0
min        230.0
25%        230.0
50%        230.0
75%        230.0
max        230.0
Name: sequence, dtype: float64
is_active
0    0.666432
1    0.333568
Name: proportion, dtype: float64
count    55386.000000
mean        -0.221483
std          0.784394
min         -2.633000
25%         -0.781000
50%         -0.358000
75%          0.236750
max          4.578000
Name: rna_dna_ratio, dtype: float64


In [ ]:
# Clean data and create train/validation split

df = pd.read_csv("dataset.tsv", sep="\t")

if "id" not in df.columns and "seq_id" in df.columns:
    df = df.rename(columns={"seq_id": "id"})

df = df.dropna(subset=["sequence", "is_active", "rna_dna_ratio"]).copy()

df["sequence"] = df["sequence"].astype(str).str.upper()
df["is_active"] = df["is_active"].astype(int)
df["rna_dna_ratio"] = df["rna_dna_ratio"].astype(float)

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["is_active"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)

print("\nTrain class distribution:")
print(train_df["is_active"].value_counts(normalize=True))

print("\nValidation class distribution:")
print(val_df["is_active"].value_counts(normalize=True))


(44308, 4) (11078, 4)
is_active
0    0.666426
1    0.333574
Name: proportion, dtype: float64
is_active
0    0.666456
1    0.333544
Name: proportion, dtype: float64


## 2. Train k-mer based classical models

These cells are the missing part that previously came from saved files in `final_model/`. They train the vectorizer, Logistic Regression, Ridge Regression, LightGBM classifier and LightGBM regressor directly in the notebook.

In [ ]:
# Helper functions for thresholds and regression scores

def find_best_thresholds(y_true, y_scores):
    thresholds = np.arange(0.05, 0.95, 0.01)

    best_acc = -1
    best_acc_t = None

    best_f1 = -1
    best_f1_t = None

    for t in thresholds:
        preds = (y_scores >= t).astype(int)

        acc = accuracy_score(y_true, preds)
        f1 = f1_score(y_true, preds)

        if acc > best_acc:
            best_acc = acc
            best_acc_t = float(t)

        if f1 > best_f1:
            best_f1 = f1
            best_f1_t = float(t)

    return best_acc, best_acc_t, best_f1, best_f1_t


def best_ratio_classification(y_true, pred_ratio, step=0.001):
    best_acc = -1
    best_t = None

    best_f1 = -1
    best_f1_t = None

    for t in np.arange(pred_ratio.min(), pred_ratio.max(), step):
        preds = (pred_ratio >= t).astype(int)

        acc = accuracy_score(y_true, preds)
        f1 = f1_score(y_true, preds)

        if acc > best_acc:
            best_acc = acc
            best_t = float(t)

        if f1 > best_f1:
            best_f1 = f1
            best_f1_t = float(t)

    return best_acc, best_t, best_f1, best_f1_t


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def prob_like_from_regression_with_stats(pred, mean, std):
    if std == 0:
        std = 1.0
    return sigmoid((pred - mean) / std)

In [ ]:
# Convert DNA sequences to k-mer count features

feature_cfg = {
    "name": "count_3_7_min2",
    "ngram_range": (3, 7),
    "binary": False,
    "min_df": 2,
}

vectorizer = CountVectorizer(
    analyzer="char",
    ngram_range=feature_cfg["ngram_range"],
    binary=feature_cfg["binary"],
    min_df=feature_cfg["min_df"]
)

X_train = vectorizer.fit_transform(train_df["sequence"])
X_val = vectorizer.transform(val_df["sequence"])

X_train_lgb = X_train.astype(np.float32)
X_val_lgb = X_val.astype(np.float32)

y_train_cls = train_df["is_active"].values
y_val_cls = val_df["is_active"].values

y_train_reg = train_df["rna_dna_ratio"].values
y_val_reg = val_df["rna_dna_ratio"].values

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))

X_train: (44308, 21824)
X_val: (11078, 21824)
Vocabulary size: 21824


In [ ]:
# Train Logistic Regression baseline

logreg_results = []
best_logreg = None
best_logreg_probs_val = None
best_logreg_result = None

for C in [0.1, 0.3, 1.0]:
    print(f"\nTraining LogisticRegression C={C}")

    logreg = LogisticRegression(
        C=C,
        max_iter=3000,
        n_jobs=-1
    )

    logreg.fit(X_train, y_train_cls)

    probs_val = logreg.predict_proba(X_val)[:, 1]

    acc, acc_t, f1, f1_t = find_best_thresholds(y_val_cls, probs_val)
    auc = roc_auc_score(y_val_cls, probs_val)

    result = {
        "C": C,
        "accuracy": acc,
        "threshold": acc_t,
        "f1": f1,
        "f1_threshold": f1_t,
        "auc": auc,
    }

    print(result)
    logreg_results.append(result)

    if best_logreg_result is None or acc > best_logreg_result["accuracy"]:
        best_logreg_result = result
        best_logreg = logreg
        best_logreg_probs_val = probs_val

print("\nBest LogReg:")
print(best_logreg_result)


Training LogisticRegression C=0.1
{'C': 0.1, 'accuracy': 0.7410182343383283, 'threshold': 0.5600000000000002, 'f1': 0.6052836918961096, 'f1_threshold': 0.23000000000000004, 'auc': np.float64(0.7738686156270569)}

Training LogisticRegression C=0.3
{'C': 0.3, 'accuracy': 0.7335259072034663, 'threshold': 0.6300000000000001, 'f1': 0.5976494634644864, 'f1_threshold': 0.12000000000000001, 'auc': np.float64(0.7599089595616745)}

Training LogisticRegression C=1.0
{'C': 1.0, 'accuracy': 0.7270265390864777, 'threshold': 0.8500000000000002, 'f1': 0.5893300248138957, 'f1_threshold': 0.1, 'auc': np.float64(0.7481794203375087)}

Best LogReg:
{'C': 0.1, 'accuracy': 0.7410182343383283, 'threshold': 0.5600000000000002, 'f1': 0.6052836918961096, 'f1_threshold': 0.23000000000000004, 'auc': np.float64(0.7738686156270569)}


In [ ]:
# Train Ridge regressor

ridge_results = []
best_ridge = None
best_ridge_pred_val = None
best_ridge_result = None

for alpha in [0.1, 1.0, 10.0]:
    print(f"\nTraining Ridge alpha={alpha}")

    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train, y_train_reg)

    pred_val = ridge.predict(X_val)

    mse = mean_squared_error(y_val_reg, pred_val)

    ratio_acc, ratio_t, ratio_f1, ratio_f1_t = best_ratio_classification(
        y_val_cls,
        pred_val
    )

    result = {
        "alpha": alpha,
        "mse": mse,
        "ratio_accuracy": ratio_acc,
        "ratio_threshold": ratio_t,
        "ratio_f1": ratio_f1,
        "ratio_f1_threshold": ratio_f1_t,
    }

    print(result)
    ridge_results.append(result)

    if best_ridge_result is None or mse < best_ridge_result["mse"]:
        best_ridge_result = result
        best_ridge = ridge
        best_ridge_pred_val = pred_val

print("\nBest Ridge:")
print(best_ridge_result)


Training Ridge alpha=0.1
{'alpha': 0.1, 'mse': 0.495540595442622, 'ratio_accuracy': 0.7374074742733345, 'ratio_threshold': 0.27625257896062827, 'ratio_f1': 0.6104910714285714, 'ratio_f1_threshold': -0.19774742103931953}

Training Ridge alpha=1.0
{'alpha': 1.0, 'mse': 0.49386148671234986, 'ratio_accuracy': 0.7381296262863333, 'ratio_threshold': 0.27755702931682613, 'ratio_f1': 0.6108506363027462, 'ratio_f1_threshold': -0.19744297068312155}

Training Ridge alpha=10.0
{'alpha': 10.0, 'mse': 0.48058385549177474, 'ratio_accuracy': 0.7402058133237046, 'ratio_threshold': 0.26635212048871004, 'ratio_f1': 0.6156202143950995, 'ratio_f1_threshold': -0.223647879511236}

Best Ridge:
{'alpha': 10.0, 'mse': 0.48058385549177474, 'ratio_accuracy': 0.7402058133237046, 'ratio_threshold': 0.26635212048871004, 'ratio_f1': 0.6156202143950995, 'ratio_f1_threshold': -0.223647879511236}


In [ ]:
# Train LightGBM classifier

lgbm_clf_configs = [
    {
        "name": "clf_leaves31_child30",
        "num_leaves": 31,
        "min_child_samples": 30,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0,
    },
    {
        "name": "clf_leaves63_child30",
        "num_leaves": 63,
        "min_child_samples": 30,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0,
    },
]

lgbm_clf_results = []
best_lgbm_clf = None
best_lgbm_clf_probs_val = None
best_lgbm_clf_result = None

for cfg in lgbm_clf_configs:
    print(f"\nTraining LightGBM classifier: {cfg['name']}")

    lgbm_clf = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=4000,
        learning_rate=0.015,
        num_leaves=cfg["num_leaves"],
        max_depth=-1,
        min_child_samples=cfg["min_child_samples"],
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=cfg["reg_alpha"],
        reg_lambda=cfg["reg_lambda"],
        random_state=42,
        n_jobs=-1,
        force_col_wise=True,
    )

    lgbm_clf.fit(
        X_train_lgb,
        y_train_cls,
        eval_set=[(X_val_lgb, y_val_cls)],
        eval_metric="binary_logloss",
        callbacks=[
            lgb.early_stopping(150),
            lgb.log_evaluation(300),
        ]
    )

    probs_val = lgbm_clf.predict_proba(X_val_lgb)[:, 1]

    acc, acc_t, f1, f1_t = find_best_thresholds(y_val_cls, probs_val)
    auc = roc_auc_score(y_val_cls, probs_val)

    result = {
        "config": cfg["name"],
        "accuracy": acc,
        "threshold": acc_t,
        "f1": f1,
        "f1_threshold": f1_t,
        "auc": auc,
        "best_iteration": int(lgbm_clf.best_iteration_),
    }

    print(result)
    lgbm_clf_results.append(result)

    if best_lgbm_clf_result is None or acc > best_lgbm_clf_result["accuracy"]:
        best_lgbm_clf_result = result
        best_lgbm_clf = lgbm_clf
        best_lgbm_clf_probs_val = probs_val

print("\nBest LightGBM classifier:")
print(best_lgbm_clf_result)



Training LightGBM classifier: clf_leaves31_child30
[LightGBM] [Info] Number of positive: 14780, number of negative: 29528
[LightGBM] [Info] Total Bins 75789
[LightGBM] [Info] Number of data points in the train set: 44308, number of used features: 21772
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.333574 -> initscore=-0.692064
[LightGBM] [Info] Start training from score -0.692064
Training until validation scores don't improve for 150 rounds
[300]	valid_0's binary_logloss: 0.53924
[600]	valid_0's binary_logloss: 0.515145
[900]	valid_0's binary_logloss: 0.50449
[1200]	valid_0's binary_logloss: 0.500195
[1500]	valid_0's binary_logloss: 0.498028
[1800]	valid_0's binary_logloss: 0.496358
[2100]	valid_0's binary_logloss: 0.494953
[2400]	valid_0's binary_logloss: 0.494148
[2700]	valid_0's binary_logloss: 0.493694
[3000]	valid_0's binary_logloss: 0.49323
[3300]	valid_0's binary_logloss: 0.492906
[3600]	valid_0's binary_logloss: 0.492658
[3900]	valid_0's binary_logloss: 0.492661
Early stop

In [ ]:
# Train LightGBM regressor

lgbm_reg_configs = [
    {
        "name": "reg_leaves31_child30",
        "num_leaves": 31,
        "min_child_samples": 30,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0,
    },
    {
        "name": "reg_leaves63_child30",
        "num_leaves": 63,
        "min_child_samples": 30,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0,
    },
]

lgbm_reg_results = []
best_lgbm_reg = None
best_lgbm_reg_pred_val = None
best_lgbm_reg_result = None

for cfg in lgbm_reg_configs:
    print(f"\nTraining LightGBM regressor: {cfg['name']}")

    lgbm_reg = lgb.LGBMRegressor(
        objective="regression",
        n_estimators=5000,
        learning_rate=0.015,
        num_leaves=cfg["num_leaves"],
        max_depth=-1,
        min_child_samples=cfg["min_child_samples"],
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=cfg["reg_alpha"],
        reg_lambda=cfg["reg_lambda"],
        random_state=42,
        n_jobs=-1,
        force_col_wise=True,
    )

    lgbm_reg.fit(
        X_train_lgb,
        y_train_reg,
        eval_set=[(X_val_lgb, y_val_reg)],
        eval_metric="l2",
        callbacks=[
            lgb.early_stopping(150),
            lgb.log_evaluation(300),
        ]
    )

    pred_val = lgbm_reg.predict(X_val_lgb)

    mse = mean_squared_error(y_val_reg, pred_val)

    ratio_acc, ratio_t, ratio_f1, ratio_f1_t = best_ratio_classification(
        y_val_cls,
        pred_val
    )

    result = {
        "config": cfg["name"],
        "mse": mse,
        "rmse": np.sqrt(mse),
        "ratio_accuracy": ratio_acc,
        "ratio_threshold": ratio_t,
        "ratio_f1": ratio_f1,
        "ratio_f1_threshold": ratio_f1_t,
        "best_iteration": int(lgbm_reg.best_iteration_),
    }

    print(result)
    lgbm_reg_results.append(result)

    if best_lgbm_reg_result is None or mse < best_lgbm_reg_result["mse"]:
        best_lgbm_reg_result = result
        best_lgbm_reg = lgbm_reg
        best_lgbm_reg_pred_val = pred_val

print("\nBest LightGBM regressor:")
print(best_lgbm_reg_result)


Training LightGBM regressor: reg_leaves31_child30
[LightGBM] [Info] Total Bins 75789
[LightGBM] [Info] Number of data points in the train set: 44308, number of used features: 21772
[LightGBM] [Info] Start training from score -0.222117
Training until validation scores don't improve for 150 rounds
[300]	valid_0's l2: 0.464679
[600]	valid_0's l2: 0.432096
[900]	valid_0's l2: 0.417636
[1200]	valid_0's l2: 0.411224
[1500]	valid_0's l2: 0.407811
[1800]	valid_0's l2: 0.405623
[2100]	valid_0's l2: 0.403934
[2400]	valid_0's l2: 0.402414
[2700]	valid_0's l2: 0.401132
[3000]	valid_0's l2: 0.400206
[3300]	valid_0's l2: 0.399673
[3600]	valid_0's l2: 0.399081
[3900]	valid_0's l2: 0.39853
[4200]	valid_0's l2: 0.397985
[4500]	valid_0's l2: 0.397555
[4800]	valid_0's l2: 0.397378
Did not meet early stopping. Best iteration is:
[4979]	valid_0's l2: 0.397149
{'config': 'reg_leaves31_child30', 'mse': 0.3971494673929755, 'rmse': np.float64(0.6301979588930573), 'ratio_accuracy': 0.7602455316844196, 'ratio_t

## 3. Train reverse-complement CNN

The CNN operates directly on one-hot encoded DNA sequences. During training, sequences are randomly replaced by their reverse complement. During validation and test-time prediction, the model averages predictions from the forward strand and reverse complement strand.

In [ ]:
# Prepare CNN data and reverse-complement utilities

import random
import copy
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, mean_squared_error

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

def reverse_complement(seq):
    complement = str.maketrans("ACGT", "TGCA")
    return seq.translate(complement)[::-1]


def one_hot_encode(seq):
    mapping = {
        "A": [1, 0, 0, 0],
        "C": [0, 1, 0, 0],
        "G": [0, 0, 1, 0],
        "T": [0, 0, 0, 1],
    }

    arr = np.array(
        [mapping.get(base, [0, 0, 0, 0]) for base in seq],
        dtype=np.float32
    )

    return arr.T

rna_mean = train_df["rna_dna_ratio"].mean()
rna_std = train_df["rna_dna_ratio"].std()

train_df_cnn = train_df.copy()
val_df_cnn = val_df.copy()

train_df_cnn["rna_scaled"] = (
    train_df_cnn["rna_dna_ratio"] - rna_mean
) / rna_std

val_df_cnn["rna_scaled"] = (
    val_df_cnn["rna_dna_ratio"] - rna_mean
) / rna_std


class DNACNNDataset(Dataset):
    def __init__(self, df, augment_rc=False, rc_prob=0.5):
        self.sequences = df["sequence"].values
        self.y_cls = df["is_active"].values.astype(np.float32)
        self.y_reg = df["rna_scaled"].values.astype(np.float32)

        self.augment_rc = augment_rc
        self.rc_prob = rc_prob

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]

        if self.augment_rc and random.random() < self.rc_prob:
            seq = reverse_complement(seq)

        x = one_hot_encode(seq)

        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(self.y_cls[idx], dtype=torch.float32),
            torch.tensor(self.y_reg[idx], dtype=torch.float32)
        )


batch_size = 128

train_cnn_dataset = DNACNNDataset(
    train_df_cnn,
    augment_rc=True,
    rc_prob=0.5
)

val_cnn_dataset = DNACNNDataset(
    val_df_cnn,
    augment_rc=False
)

pin_memory = torch.cuda.is_available()

train_cnn_loader = DataLoader(
    train_cnn_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=pin_memory
)

val_cnn_loader = DataLoader(
    val_cnn_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=pin_memory
)

print("Train batches:", len(train_cnn_loader))
print("Val batches:", len(val_cnn_loader))

Using device: cuda
Train batches: 347
Val batches: 87


In [ ]:
# Define CNN architecture

class ReverseComplementCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv1d(4, 96, kernel_size=9, padding=4),
            nn.BatchNorm1d(96),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.20),

            nn.Conv1d(96, 160, kernel_size=7, padding=3),
            nn.BatchNorm1d(160),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.25),

            nn.Conv1d(160, 256, kernel_size=5, padding=2),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.25),

            nn.Conv1d(256, 256, kernel_size=5, padding=2),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.20),
        )

        self.global_max_pool = nn.AdaptiveMaxPool1d(1)
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)

        self.shared = nn.Sequential(
            nn.Linear(256 * 2, 256),
            nn.ReLU(),
            nn.Dropout(0.25),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.20)
        )

        self.classification_head = nn.Linear(128, 1)
        self.regression_head = nn.Linear(128, 1)

    def forward(self, x):
        x = self.features(x)

        x_max = self.global_max_pool(x).squeeze(-1)
        x_avg = self.global_avg_pool(x).squeeze(-1)

        x = torch.cat([x_max, x_avg], dim=1)
        x = self.shared(x)

        cls_logits = self.classification_head(x).squeeze(1)
        reg_output = self.regression_head(x).squeeze(1)

        return cls_logits, reg_output

model_rc = ReverseComplementCNN().to(device)
print(model_rc)

ReverseComplementCNN(
  (features): Sequential(
    (0): Conv1d(4, 96, kernel_size=(9,), stride=(1,), padding=(4,))
    (1): BatchNorm1d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Dropout(p=0.2, inplace=False)
    (5): Conv1d(96, 160, kernel_size=(7,), stride=(1,), padding=(3,))
    (6): BatchNorm1d(160, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): ReLU()
    (8): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Dropout(p=0.25, inplace=False)
    (10): Conv1d(160, 256, kernel_size=(5,), stride=(1,), padding=(2,))
    (11): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (12): ReLU()
    (13): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (14): Dropout(p=0.3, inplace=False)
    (15): Conv1d(256, 256, kernel_size=(5,), stride

In [ ]:
# Define CNN losses and evaluation functions

n_pos = train_df_cnn["is_active"].sum()
n_neg = len(train_df_cnn) - n_pos

pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(device)

criterion_cls = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print("pos_weight:", pos_weight.item())

criterion_reg = nn.HuberLoss(delta=1.0)

alpha_reg = 0.3


def find_best_thresholds(y_true, y_scores):
    thresholds = np.arange(0.05, 0.95, 0.01)

    best_acc = -1
    best_acc_t = None
    best_f1 = -1
    best_f1_t = None

    for t in thresholds:
        preds = (y_scores >= t).astype(int)

        acc = accuracy_score(y_true, preds)
        f1 = f1_score(y_true, preds)

        if acc > best_acc:
            best_acc = acc
            best_acc_t = float(t)

        if f1 > best_f1:
            best_f1 = f1
            best_f1_t = float(t)

    return best_acc, best_acc_t, best_f1, best_f1_t


def predict_batch(model, sequences, device):
    x = np.stack([one_hot_encode(seq) for seq in sequences])
    x = torch.tensor(x, dtype=torch.float32).to(device)

    cls_logits, reg_scaled = model(x)

    probs = torch.sigmoid(cls_logits).detach().cpu().numpy()
    reg_scaled = reg_scaled.detach().cpu().numpy()

    return probs, reg_scaled


def evaluate_cnn(model, df, batch_size=512, use_rc_tta=True):
    model.eval()

    all_probs = []
    all_reg_preds = []

    sequences = df["sequence"].tolist()

    with torch.no_grad():
        for start in range(0, len(sequences), batch_size):
            batch_seqs = sequences[start:start + batch_size]

            probs_forward, reg_forward = predict_batch(model, batch_seqs, device)

            if use_rc_tta:
                batch_rc = [reverse_complement(seq) for seq in batch_seqs]
                probs_rc, reg_rc = predict_batch(model, batch_rc, device)

                probs = (probs_forward + probs_rc) / 2
                reg_scaled = (reg_forward + reg_rc) / 2
            else:
                probs = probs_forward
                reg_scaled = reg_forward

            reg_original = reg_scaled * rna_std + rna_mean

            all_probs.extend(probs)
            all_reg_preds.extend(reg_original)

    all_probs = np.array(all_probs)
    all_reg_preds = np.array(all_reg_preds)

    y_true_cls = df["is_active"].values
    y_true_reg = df["rna_dna_ratio"].values

    best_acc, best_acc_t, best_f1, best_f1_t = find_best_thresholds(
        y_true_cls,
        all_probs
    )

    auc = roc_auc_score(y_true_cls, all_probs)
    mse = mean_squared_error(y_true_reg, all_reg_preds)

    return {
        "probs": all_probs,
        "reg_preds": all_reg_preds,
        "best_acc": best_acc,
        "best_acc_threshold": best_acc_t,
        "best_f1": best_f1,
        "best_f1_threshold": best_f1_t,
        "auc": auc,
        "mse": mse
    }

In [ ]:
# Compute baseline MSE for the regression target

mean_pred = np.full_like(
    val_df_cnn["rna_dna_ratio"].values,
    fill_value=train_df_cnn["rna_dna_ratio"].mean(),
    dtype=float
)

baseline_mse = mean_squared_error(
    val_df_cnn["rna_dna_ratio"].values,
    mean_pred
)

print("Baseline MSE:", baseline_mse)

Baseline MSE: 0.6170852441710788


In [ ]:
# Train CNN and keep the best validation model

optimizer = torch.optim.AdamW(
    model_rc.parameters(),
    lr=3e-4,
    weight_decay=5e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3
)

num_epochs = 20
patience = 5

best_state_rc = None
best_info_rc = None
best_score_rc = -float("inf")
epochs_without_improvement = 0

for epoch in range(num_epochs):
    model_rc.train()

    total_loss = 0
    total_batches = 0

    print(f"\nEpoch {epoch + 1}/{num_epochs}")

    for batch_idx, (x, y_cls, y_reg) in enumerate(train_cnn_loader):
        x = x.to(device)
        y_cls = y_cls.to(device)
        y_reg = y_reg.to(device)

        optimizer.zero_grad()

        cls_logits, reg_scaled_pred = model_rc(x)

        loss_cls = criterion_cls(cls_logits, y_cls)
        loss_reg = criterion_reg(reg_scaled_pred, y_reg)

        loss = loss_cls + alpha_reg * loss_reg

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_rc.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        total_batches += 1

        if batch_idx % 100 == 0:
            print(
                f"Batch {batch_idx}/{len(train_cnn_loader)} | "
                f"Loss: {loss.item():.4f} | "
                f"Cls: {loss_cls.item():.4f} | "
                f"Reg: {loss_reg.item():.4f}"
            )

    train_loss = total_loss / total_batches

    val_no_tta = evaluate_cnn(model_rc, val_df_cnn, use_rc_tta=False)
    val_tta = evaluate_cnn(model_rc, val_df_cnn, use_rc_tta=True)
    auc_score = 2 * val_tta["auc"] - 1
    mse_score = 1 - val_tta["mse"] / baseline_mse

    score = 0.7 * auc_score + 0.3 * mse_score

    print("AUC score:", auc_score)
    print("MSE score:", mse_score)
    print("Combined score:", score)

    scheduler.step(score)

    print(f"Train loss: {train_loss:.4f}")

    print("\nValidation without RC TTA:")
    print("Acc:", val_no_tta["best_acc"])
    print("F1:", val_no_tta["best_f1"])
    print("AUC:", val_no_tta["auc"])
    print("MSE:", val_no_tta["mse"])

    print("\nValidation with RC TTA:")
    print("Acc:", val_tta["best_acc"])
    print("Threshold:", val_tta["best_acc_threshold"])
    print("F1:", val_tta["best_f1"])
    print("AUC:", val_tta["auc"])
    print("MSE:", val_tta["mse"])
    print("Score:", score)

    scheduler.step(val_tta["mse"])

    if score > best_score_rc:
        best_score_rc = score
        best_state_rc = copy.deepcopy(model_rc.state_dict())
        best_info_rc = {
            "epoch": epoch + 1,
            "best_acc": val_tta["best_acc"],
            "best_acc_threshold": val_tta["best_acc_threshold"],
            "best_f1": val_tta["best_f1"],
            "best_f1_threshold": val_tta["best_f1_threshold"],
            "auc": val_tta["auc"],
            "mse": val_tta["mse"],
            "score": score
        }
        epochs_without_improvement = 0
        print("New best RC CNN saved.")
    else:
        epochs_without_improvement += 1
        print(f"No improvement for {epochs_without_improvement} epoch(s).")

    if epochs_without_improvement >= patience:
        print("Early stopping.")
        break

print("\nBest RC CNN:")
print(best_info_rc)


Epoch 1/20
Batch 0/347 | Loss: 0.6743 | Cls: 0.4685 | Reg: 0.2940
Batch 100/347 | Loss: 0.6344 | Cls: 0.4492 | Reg: 0.2646
Batch 200/347 | Loss: 0.6857 | Cls: 0.5300 | Reg: 0.2224
Batch 300/347 | Loss: 0.6972 | Cls: 0.5270 | Reg: 0.2432
AUC score: 0.62827994018369
MSE score: 0.23348195390567428
Combined score: 0.5098405443002854
Train loss: 0.6999

Validation without RC TTA:
Acc: 0.7566347716194259
F1: 0.6430249946592609
AUC: 0.8085378269978741
MSE: 0.4752861376139775

Validation with RC TTA:
Acc: 0.7590720346632966
Threshold: 0.37000000000000005
F1: 0.6464964298680866
AUC: 0.814139970091845
MSE: 0.4730069756356552
Score: 0.5098405443002854
New best RC CNN saved.

Epoch 2/20
Batch 0/347 | Loss: 0.7672 | Cls: 0.5586 | Reg: 0.2980
Batch 100/347 | Loss: 0.7885 | Cls: 0.5522 | Reg: 0.3376
Batch 200/347 | Loss: 0.6816 | Cls: 0.4902 | Reg: 0.2735
Batch 300/347 | Loss: 0.6566 | Cls: 0.4977 | Reg: 0.2269
AUC score: 0.6233214327542134
MSE score: 0.16257298645339213
Combined score: 0.4850968988

In [ ]:
# Evaluate CNN on validation data

val_rc_current = evaluate_cnn(
    model_rc,
    val_df_cnn,
    use_rc_tta=True
)

print("Current / last RC CNN:")
print("Accuracy:", val_rc_current["best_acc"])
print("Threshold:", val_rc_current["best_acc_threshold"])
print("F1:", val_rc_current["best_f1"])
print("F1 threshold:", val_rc_current["best_f1_threshold"])
print("AUC:", val_rc_current["auc"])
print("MSE:", val_rc_current["mse"])

cnn_rc_probs = val_rc_current["probs"]

Current / last RC CNN:
Accuracy: 0.766835168803033
Threshold: 0.29000000000000004
F1: 0.6564376250833889
F1 threshold: 0.16000000000000003
AUC: 0.824587186633815
MSE: 0.4917826969700659


## 4. Build the final ensemble

Here the validation predictions from the trained k-mer models are combined with the CNN probabilities. Regression outputs are standardized and transformed with a sigmoid to create probability-like classification signals.

In [ ]:
# Prepare model predictions for the ensemble

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

logreg = best_logreg
ridge = best_ridge
lgbm_clf = best_lgbm_clf
lgbm_reg = best_lgbm_reg

logreg_probs = best_logreg_probs_val
lgbm_clf_probs = best_lgbm_clf_probs_val

ridge_pred = best_ridge_pred_val
lgbm_reg_pred = best_lgbm_reg_pred_val
ridge_pred_mean = float(ridge_pred.mean())
ridge_pred_std = float(ridge_pred.std())

lgbm_reg_pred_mean = float(lgbm_reg_pred.mean())
lgbm_reg_pred_std = float(lgbm_reg_pred.std())

ridge_prob_like = sigmoid((ridge_pred - ridge_pred_mean) / ridge_pred_std)
lgbm_reg_prob_like = sigmoid((lgbm_reg_pred - lgbm_reg_pred_mean) / lgbm_reg_pred_std)

cnn_probs = val_rc_current["probs"]

y_true = val_df["is_active"].values

print("Prepared predictions from trained models.")
print("CNN probs shape:", cnn_probs.shape)
print("LightGBM classifier probs shape:", lgbm_clf_probs.shape)
print("LogReg probs shape:", logreg_probs.shape)
print("Ridge score shape:", ridge_prob_like.shape)
print("LightGBM regressor score shape:", lgbm_reg_prob_like.shape)
print("Validation labels shape:", y_true.shape)

In [ ]:
# Search ensemble weights and classification threshold

best_acc = -1
best_f1 = -1

best_acc_config = None
best_f1_config = None

weight_step = 0.1
scale = int(round(1 / weight_step))

for i in range(scale + 1):
    for j in range(scale + 1 - i):
        for k in range(scale + 1 - i - j):
            for l in range(scale + 1 - i - j - k):

                m = scale - i - j - k - l

                w_cnn = i / scale
                w_lgbm_clf = j / scale
                w_logreg = k / scale
                w_lgbm_reg = l / scale
                w_ridge = m / scale

                probs = (
                    w_cnn * cnn_probs
                    + w_lgbm_clf * lgbm_clf_probs
                    + w_logreg * logreg_probs
                    + w_lgbm_reg * lgbm_reg_prob_like
                    + w_ridge * ridge_prob_like
                )

                for t in np.arange(0.05, 0.95, 0.01):
                    preds = (probs >= t).astype(int)

                    acc = accuracy_score(y_true, preds)
                    f1 = f1_score(y_true, preds)

                    if acc > best_acc:
                        best_acc = acc
                        best_acc_config = {
                            "w_cnn": float(w_cnn),
                            "w_lgbm_clf": float(w_lgbm_clf),
                            "w_logreg": float(w_logreg),
                            "w_lgbm_reg": float(w_lgbm_reg),
                            "w_ridge": float(w_ridge),
                            "threshold": float(t),
                            "accuracy": float(acc),
                            "f1": float(f1),
                            "auc": float(roc_auc_score(y_true, probs))
                        }

                    if f1 > best_f1:
                        best_f1 = f1
                        best_f1_config = {
                            "w_cnn": float(w_cnn),
                            "w_lgbm_clf": float(w_lgbm_clf),
                            "w_logreg": float(w_logreg),
                            "w_lgbm_reg": float(w_lgbm_reg),
                            "w_ridge": float(w_ridge),
                            "threshold": float(t),
                            "accuracy": float(acc),
                            "f1": float(f1),
                            "auc": float(roc_auc_score(y_true, probs))
                        }

print("Best accuracy config:")
print(best_acc_config)

print("\nBest F1 config:")
print(best_f1_config)

Best accuracy config:
{'w_cnn': 0.5, 'w_lgbm_clf': 0.2, 'w_logreg': 0.0, 'w_lgbm_reg': 0.0, 'w_ridge': 0.3, 'threshold': 0.4100000000000001, 'accuracy': 0.7813684780646326, 'f1': 0.6254253015774822, 'auc': 0.8386383376798948}

Best F1 config:
{'w_cnn': 0.6, 'w_lgbm_clf': 0.1, 'w_logreg': 0.0, 'w_lgbm_reg': 0.2, 'w_ridge': 0.1, 'threshold': 0.30000000000000004, 'accuracy': 0.7516699765300596, 'f1': 0.6742451154529308, 'auc': 0.8418624727068382}


In [ ]:
# Evaluate the selected final ensemble

final_ensemble_config = best_acc_config

final_probs = (
    final_ensemble_config["w_cnn"] * cnn_probs
    + final_ensemble_config["w_lgbm_clf"] * lgbm_clf_probs
    + final_ensemble_config["w_logreg"] * logreg_probs
    + final_ensemble_config["w_lgbm_reg"] * lgbm_reg_prob_like
    + final_ensemble_config["w_ridge"] * ridge_prob_like
)

final_preds = (final_probs >= final_ensemble_config["threshold"]).astype(int)

final_acc = accuracy_score(y_true, final_preds)
final_f1 = f1_score(y_true, final_preds)
final_auc = roc_auc_score(y_true, final_probs)

final_lgbm_reg_mse = mean_squared_error(val_df["rna_dna_ratio"].values, lgbm_reg_pred)
final_lgbm_reg_rmse = np.sqrt(final_lgbm_reg_mse)

final_ridge_mse = mean_squared_error(val_df["rna_dna_ratio"].values, ridge_pred)

print("FINAL VALIDATION RESULTS")
print("Accuracy:", final_acc)
print("F1:", final_f1)
print("AUC:", final_auc)
print("LightGBM regressor MSE:", final_lgbm_reg_mse)
print("LightGBM regressor RMSE:", final_lgbm_reg_rmse)
print("Ridge MSE:", final_ridge_mse)
print("\nFinal ensemble config:")
print(final_ensemble_config)
print("\nConfusion matrix:")
print(confusion_matrix(y_true, final_preds))
print("\nClassification report:")
print(classification_report(y_true, final_preds))

## 5. Save the final model

All model components are saved into `final_model/`. This folder is what `evaluation_script.py` expects at inference time.

In [ ]:
# Save trained models and final configuration

MODEL_DIR = "final_model"
os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(vectorizer, os.path.join(MODEL_DIR, "kmer_vectorizer.pkl"))
joblib.dump(logreg, os.path.join(MODEL_DIR, "logreg_classifier.pkl"))
joblib.dump(ridge, os.path.join(MODEL_DIR, "ridge_regressor.pkl"))
joblib.dump(lgbm_clf, os.path.join(MODEL_DIR, "lgbm_classifier.pkl"))
joblib.dump(lgbm_reg, os.path.join(MODEL_DIR, "lgbm_regressor.pkl"))

cnn_info = {
    "model_type": "ReverseComplementCNN",
    "classification_accuracy": float(val_rc_current["best_acc"]),
    "classification_threshold": float(val_rc_current["best_acc_threshold"]),
    "f1": float(val_rc_current["best_f1"]),
    "auc": float(val_rc_current["auc"]),
    "mse": float(val_rc_current["mse"]),
    "uses_reverse_complement_tta": True,
}

torch.save(
    {
        "model_state_dict": model_rc.state_dict(),
        "rna_mean": float(rna_mean),
        "rna_std": float(rna_std),
        "cnn_info": cnn_info,
    },
    os.path.join(MODEL_DIR, "rc_cnn_model.pth")
)

with open(os.path.join(MODEL_DIR, "rc_cnn_info.json"), "w") as f:
    json.dump(cnn_info, f, indent=2)

# Save config used by evaluation_script.py
final_config = {
    "use_full_sequence": True,
    "feature_config": {
        "name": feature_cfg["name"],
        "ngram_range": list(feature_cfg["ngram_range"]),
        "binary": feature_cfg["binary"],
        "min_df": feature_cfg["min_df"],
    },
    "classification_model": "cnn_lgbm_ridge_ensemble",
    "regression_model": "lightgbm_regressor",

    "w_cnn": float(final_ensemble_config["w_cnn"]),
    "w_lgbm_clf": float(final_ensemble_config["w_lgbm_clf"]),
    "w_logreg": float(final_ensemble_config["w_logreg"]),
    "w_lgbm_reg": float(final_ensemble_config["w_lgbm_reg"]),
    "w_ridge": float(final_ensemble_config["w_ridge"]),

    "classification_threshold": float(final_ensemble_config["threshold"]),

    "validation_accuracy": float(final_acc),
    "validation_f1": float(final_f1),
    "validation_auc": float(final_auc),
    "validation_lgbm_reg_mse": float(final_lgbm_reg_mse),
    "validation_ridge_mse": float(final_ridge_mse),

    "ridge_pred_mean": float(ridge_pred_mean),
    "ridge_pred_std": float(ridge_pred_std),
    "lgbm_reg_pred_mean": float(lgbm_reg_pred_mean),
    "lgbm_reg_pred_std": float(lgbm_reg_pred_std),

    "uses_cnn": True,
}

with open(os.path.join(MODEL_DIR, "final_config.json"), "w") as f:
    json.dump(final_config, f, indent=2)

print("Saved final model files to:", MODEL_DIR)
print(json.dumps(final_config, indent=2))